# 01. Preprocessing: Grain Segmentation with Cellpose

This notebook demonstrates the preprocessing pipeline used to turn a bulk
photograph of sake rice grains into individually segmented, rotation-corrected,
background-masked crops suitable for downstream classification.

Pipeline stages (see `src/sake_rice_inspection/preprocessing.py`):
1. **Segmentation** — Cellpose separates touching grains into instance masks.
2. **Noise reduction** — an area filter (default 1500 px) discards fragments
   that are too small to be a real grain.
3. **Normalization** — each grain is rotated so its major axis is vertical
   (via `regionprops` orientation) and the background is masked out, isolating
   texture information for the classifier.

> **Note on data**: the original grain photographs used in this research are
> subject to an NDA with the data-providing institutions and are not included
> in this repository. Point `CURATED_ROOT` below at your own directory of
> labeled grain photographs (one subfolder per class) to run this notebook.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

from cellpose import models

from sake_rice_inspection.preprocessing import process_directory, process_image_to_grains, render_overlay

## Single-trait grains

Each label subfolder under `CURATED_ROOT` contains photographs of grains
exhibiting a single visual trait. `process_directory` walks every subfolder,
segments each photograph, and writes one crop + mask per grain.

In [ ]:
cp_model = models.CellposeModel(gpu=True)

CURATED_ROOT = Path("../data/sample/reflect/single")
OUTPUT_ROOT = Path("../outputs/crops")
MASK_OUTPUT = Path("../outputs/mask")
CROP_MASK_OUTPUT = Path("../outputs/crops_mask")

stats = process_directory(
    CURATED_ROOT, OUTPUT_ROOT, MASK_OUTPUT, CROP_MASK_OUTPUT,
    model=cp_model, grouped_by_image=False, min_area=1500, diameter=50, margin=20,
)
stats

## Composite-trait grains

Grains that exhibit multiple co-occurring traits (e.g. both Shinpaku and Base
White) are photographed and stored differently: each source photo gets its
own subfolder so its grains don't collide in ID with grains from other
photos in the same label. Pass `grouped_by_image=True` to get this layout.

In [ ]:
CURATED_ROOT_COMPOSITE = Path("../data/sample/reflect/composite")
OUTPUT_ROOT_COMPOSITE = Path("../outputs/crops_composite")
MASK_OUTPUT_COMPOSITE = Path("../outputs/mask_composite")
CROP_MASK_OUTPUT_COMPOSITE = Path("../outputs/crops_mask_composite")

stats_composite = process_directory(
    CURATED_ROOT_COMPOSITE, OUTPUT_ROOT_COMPOSITE, MASK_OUTPUT_COMPOSITE, CROP_MASK_OUTPUT_COMPOSITE,
    model=cp_model, grouped_by_image=True, min_area=1500, diameter=50, margin=20,
)
stats_composite

## Inspecting a single image

For debugging or a quick sanity check on one photograph, `process_image_to_grains`
runs the same segmentation + extraction logic without writing any files, and
`render_overlay` produces a visualization of the instance mask.

In [ ]:
import matplotlib.pyplot as plt
from skimage import io as skio

from sake_rice_inspection.preprocessing import segment_image

sample_image_path = Path("../data/sample/reflect/single/example_label/example.jpg")

image = skio.imread(sample_image_path)
mask, region_df = segment_image(cp_model, image, min_area=1500, diameter=50)

plt.figure(figsize=(8, 8))
plt.imshow(render_overlay(image, mask))
plt.title(f"Detected grains: {mask.max()}")
plt.axis("off")
plt.show()

grains = process_image_to_grains(sample_image_path, cp_model, min_area=1500, diameter=50, margin=20)
print(f"Extracted {len(grains)} individual grain crops")